# E8^k HAWK-Style Signing Stage

This notebook works as follows: generate a secret basis `B_secret` for `E8^k`, publish only its Gram matrix `Q = B_secret * B_secret^T`, hash `(message, salt, Q)` to a binary challenge `h in {0,1}^{8k}`, form the hidden half-lattice target `(h * B_secret) / 2`, and use the `b = 2` bootstrap sampler. To sample the half shift at width `sigma`, the signer samples an integer representative from `(h * B_secret) + 2E8^k` at width `2 * sigma` and then halves it.

The returned signature coordinates satisfy `w = h - 2s`, where `w * B_secret` is the representative selected from the sampled plus/minus pair by the same sign-canonicalization idea as HAWK's `sym-break`: if the first nonzero coordinate of `w` is negative, replace `w` by `-w` before deriving `s`. The signer now also rejects samples with public norm `w * Q * w^T = ||w * B_secret||^2` above the HAWK-style threshold `4 * dimension * sigma_verify^2`. This is still an experimental E8 signing stage, not a full HAWK implementation.


In [7]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler as DGaussZ
from sage.all import vector, matrix, ZZ, QQ, codes
from mpmath import mp, exp, jtheta, sqrt, pi, log

import itertools
import random
import hashlib
import os

mp.dps = 80

In [8]:
C = codes.BinaryReedMullerCode(1, 3)
CODEWORD_TUPLES = [tuple(int(x) for x in cw) for cw in C]
CODEWORD_SET = set(CODEWORD_TUPLES)

# Construction A setup for the E8 model used by sampler:
# E8 = RM(1,3) + 2 Z^8, up to scaling.
I2 = [vector(ZZ, [2 if i == j else 0 for j in range(8)]) for i in range(8)]
G = C.generator_matrix()
Grows = [vector(ZZ, [int(x) for x in G.row(i)]) for i in range(G.nrows())]
E8_BASIS = matrix(ZZ, I2 + Grows).row_module().basis_matrix()
E8_PUBLIC_GRAM_BLOCK = E8_BASIS * E8_BASIS.transpose()


def e8_mod_2e8_representatives():
    reps = []
    for bits in itertools.product([0, 1], repeat=8):
        r = vector(ZZ, [0] * 8)
        for i, b in enumerate(bits):
            if b:
                r += E8_BASIS.row(i)
        reps.append(r)
    return reps


def direct_sum_e8_basis(k):
    if k <= 0:
        raise ValueError("k must be positive")
    dim = E8_BASIS.nrows()
    basis = matrix(ZZ, dim * k, dim * k)
    for block in range(k):
        offset = block * dim
        for i in range(dim):
            for j in range(dim):
                basis[offset + i, offset + j] = E8_BASIS[i, j]
    return basis


def public_key_gram_matrix(k, basis=None):
    if basis is None:
        basis = direct_sum_e8_basis(k)
    return basis * basis.transpose()


def serialize_integer_matrix(mat):
    rows = []
    for i in range(mat.nrows()):
        rows.append(",".join(str(ZZ(mat[i, j])) for j in range(mat.ncols())))
    return (";".join(rows)).encode("ascii")


def public_key_gram_descriptor(k):
    # For E8^k this describes the block diagonal public Gram matrix.
    return (
        b"E8-direct-sum-gram-v1"
        + int(k).to_bytes(4, "big")
        + serialize_integer_matrix(E8_PUBLIC_GRAM_BLOCK)
    )


def public_key_gram_descriptor_from_matrix(k, public_gram):
    return (
        b"E8-secret-basis-gram-v1"
        + int(k).to_bytes(4, "big")
        + serialize_integer_matrix(public_gram)
    )


def identity_matrix_zz(dim):
    return matrix(ZZ, [[1 if i == j else 0 for j in range(dim)] for i in range(dim)])


def random_block_unimodular_matrix(k, steps_per_block=24, coeff_choices=(-1, 1)):
    dim = 8 * k
    U = identity_matrix_zz(dim)
    for block in range(k):
        offset = 8 * block
        block_indices = list(range(offset, offset + 8))
        for _ in range(steps_per_block):
            i, j = random.sample(block_indices, 2)
            c = random.choice(coeff_choices)
            U[i, :] = U[i, :] + c * U[j, :]
        for i in block_indices:
            if random.choice([False, True]):
                U[i, :] = -U[i, :]
    return U


def generate_e8_hawk_style_keypair(k, steps_per_block=24):
    base_basis = direct_sum_e8_basis(k)
    secret_mixing = random_block_unimodular_matrix(k, steps_per_block=steps_per_block)
    secret_basis = secret_mixing * base_basis
    public_gram = public_key_gram_matrix(k, secret_basis)
    public_key = {
        "k": k,
        "Q": public_gram,
        "bytes": public_key_gram_descriptor_from_matrix(k, public_gram),
    }
    secret_key = {
        "k": k,
        "basis": secret_basis,
        "mixing": secret_mixing,
        "public_key": public_key,
    }
    return {"public_key": public_key, "secret_key": secret_key}


REPS = e8_mod_2e8_representatives()
print("single-block quotient size:", len(REPS))
print("E8 basis shape:", E8_BASIS.nrows(), "x", E8_BASIS.ncols())
print("single-block public Gram shape:", E8_PUBLIC_GRAM_BLOCK.nrows(), "x", E8_PUBLIC_GRAM_BLOCK.ncols())


single-block quotient size: 256
E8 basis shape: 8 x 8
single-block public Gram shape: 8 x 8


In [9]:
def nome(s):
    return exp(-pi / (s**2))


def rho_2Z_shift(alpha, s):
    q = nome(s)
    z = -2j * pi * alpha / (s**2)
    value = (q ** (alpha**2)) * jtheta(3, z, q**4)
    return float(value.real)


def codeword_weights_shifted_E8(s, shift):
    shift_f = [float(a) for a in shift]
    log_weights = []
    for c in CODEWORD_TUPLES:
        lw = 0.0
        for i in range(8):
            lw += float(log(rho_2Z_shift(c[i] + shift_f[i], s)))
        log_weights.append(lw)
    m = max(log_weights)
    return [float(exp(w - m)) for w in log_weights]


def sample_coord_2Z_coset(alpha, s):
    sigma = float(s) / (2 * float(sqrt(2 * pi)))
    center = -QQ(alpha) / 2
    z = DGaussZ(sigma=sigma, c=center)()
    return ZZ(2 * z) + QQ(alpha)


def sample_shifted_E8(s, shift):
    if len(shift) != 8:
        raise ValueError("shift must have length 8")

    weights = codeword_weights_shifted_E8(s, shift)
    idx = random.choices(range(len(CODEWORD_TUPLES)), weights=weights, k=1)[0]
    c = CODEWORD_TUPLES[idx]
    x = [sample_coord_2Z_coset(c[i] + shift[i], s) for i in range(8)]
    return vector(QQ, x), c


def sample_coset_rep_plus_2E8(s, rep):
    # Bootstrap identity: D_{rep + 2E8, s} = 2 * D_{E8 + rep/2, s/2}.
    shift = [QQ(rep[i]) / 2 for i in range(8)]
    y, _ = sample_shifted_E8(s / 2, shift)
    return vector(QQ, [2 * yi for yi in y])


def in_E8(v):
    if len(v) != 8:
        return False
    for x in v:
        if x not in ZZ:
            return False
    parity = tuple(int(ZZ(x) % 2) for x in v)
    return parity in CODEWORD_SET


def in_2E8(v):
    if len(v) != 8:
        return False
    if any(ZZ(x) % 2 != 0 for x in v):
        return False
    half = vector(ZZ, [ZZ(x) // 2 for x in v])
    return in_E8(half)


def in_rep_plus_2E8(v, rep):
    if len(v) != len(rep):
        return False
    diff = vector(ZZ, [ZZ(v[i] - rep[i]) for i in range(8)])
    return in_2E8(diff)


def split_e8_blocks(v):
    if len(v) % 8 != 0:
        raise ValueError("vector length must be a multiple of 8")
    return [vector(v.base_ring(), list(v[8 * i:8 * (i + 1)])) for i in range(len(v) // 8)]


def sample_direct_sum_coset_rep_plus_2E8(s, coset_rep):
    blocks = split_e8_blocks(coset_rep)
    samples = [sample_coset_rep_plus_2E8(s, block) for block in blocks]
    out = []
    for sample in samples:
        out.extend(sample)
    return vector(QQ, out)


def in_direct_sum_E8(v):
    return all(in_E8(block) for block in split_e8_blocks(v))


def in_direct_sum_rep_plus_2E8(v, coset_rep):
    if len(v) != len(coset_rep):
        return False
    v_blocks = split_e8_blocks(v)
    rep_blocks = split_e8_blocks(coset_rep)
    return all(in_rep_plus_2E8(vb, rb) for vb, rb in zip(v_blocks, rep_blocks))


def lattice_point_to_basis_coefficients(v, basis):
    v_q = vector(QQ, v)
    coeffs_q = basis.transpose().solve_right(v_q.column()).column(0)
    if any(c not in ZZ for c in coeffs_q):
        raise ValueError("lattice point does not have integral coordinates in the chosen basis")
    return vector(ZZ, [ZZ(c) for c in coeffs_q])


In [10]:
def _normalize_binary_message(msg_bits):
    if isinstance(msg_bits, str):
        if any(ch not in "01" for ch in msg_bits):
            raise ValueError("bitstring must only contain '0' and '1'")
        return msg_bits

    try:
        bits = []
        for b in msg_bits:
            if int(b) not in (0, 1):
                raise ValueError
            bits.append(str(int(b)))
        return "".join(bits)
    except Exception as exc:
        raise ValueError("msg_bits must be a bitstring or iterable of bits") from exc


def _pack_bitstring(bitstring):
    bitlen = len(bitstring)
    if bitlen == 0:
        return 0, b""
    pad = (-bitlen) % 8
    padded = bitstring + ("0" * pad)
    value = int(padded, 2)
    return bitlen, value.to_bytes(len(padded) // 8, "big")


def _normalize_salt(salt):
    if isinstance(salt, bytes):
        return salt
    if isinstance(salt, str):
        s = salt[2:] if salt.startswith("0x") else salt
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("salt hex string is invalid") from exc
    raise ValueError("salt must be bytes or hex string")


def _normalize_pubkey(pubkey_bytes):
    if pubkey_bytes is None:
        return b""
    if isinstance(pubkey_bytes, bytes):
        return pubkey_bytes
    if isinstance(pubkey_bytes, str):
        s = pubkey_bytes[2:] if pubkey_bytes.startswith("0x") else pubkey_bytes
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("pubkey hex string is invalid") from exc
    raise ValueError("pubkey_bytes must be None, bytes, or hex string")


def build_hash_transcript(msg_bits, salt, pubkey_bytes=None, domain_sep=b"E8-hawk-binary-challenge-v1"):
    bitstring = _normalize_binary_message(msg_bits)
    bitlen, packed = _pack_bitstring(bitstring)
    salt_b = _normalize_salt(salt)
    pub_b = _normalize_pubkey(pubkey_bytes)
    transcript = (
        domain_sep
        + len(pub_b).to_bytes(4, "big") + pub_b
        + len(salt_b).to_bytes(2, "big") + salt_b
        + bitlen.to_bytes(8, "big") + packed
    )
    return transcript, salt_b


def _shake_256_bits(transcript, bit_count):
    if bit_count <= 0:
        raise ValueError("bit_count must be positive")
    digest = hashlib.shake_256(transcript).digest((bit_count + 7) // 8)
    bits = []
    for byte in digest:
        for shift in range(7, -1, -1):
            bits.append(ZZ((byte >> shift) & 1))
            if len(bits) == bit_count:
                return vector(ZZ, bits)
    raise RuntimeError("not enough XOF output")


def hash_to_binary_challenge(msg_bits, salt, pubkey_bytes=None, k=1):
    basis = direct_sum_e8_basis(k)
    transcript, salt_b = build_hash_transcript(msg_bits, salt, pubkey_bytes=pubkey_bytes)
    h = _shake_256_bits(transcript, basis.nrows())
    return h, basis, salt_b


def challenge_to_lattice_shift(h, basis):
    h_vec = vector(ZZ, h)
    if len(h_vec) != basis.nrows():
        raise ValueError("challenge length must match the number of basis rows")
    if any(bit not in (ZZ(0), ZZ(1)) for bit in h_vec):
        raise ValueError("challenge h must be binary")
    coset_rep = vector(ZZ, h_vec * basis)
    half_shift = vector(QQ, [QQ(x) / 2 for x in coset_rep])
    return coset_rep, half_shift


def hash_message_to_hawk_shift(msg_bits, salt=None, pubkey_bytes=None, k=1):
    if salt is None:
        salt_b = os.urandom(16)
    else:
        salt_b = _normalize_salt(salt)

    h, basis, salt_b = hash_to_binary_challenge(
        msg_bits,
        salt_b,
        pubkey_bytes=pubkey_bytes,
        k=k,
    )
    coset_rep, half_shift = challenge_to_lattice_shift(h, basis)
    return {
        "salt": salt_b,
        "k": k,
        "b": 2,
        "basis": basis,
        "h": h,
        "coset_rep": coset_rep,
        "half_shift": half_shift,
        "challenge_is_binary": all(bit in (ZZ(0), ZZ(1)) for bit in h),
        "half_shift_doubles_to_rep": vector(ZZ, [ZZ(2 * x) for x in half_shift]) == coset_rep,
    }


In [11]:
def sym_break_vector(w):
    # HAWK sym-break analogue: the first nonzero coordinate is positive.
    for wi in w:
        wi = ZZ(wi)
        if wi > 0:
            return True
        if wi < 0:
            return False
    return False


def squared_euclidean_norm(v):
    return sum(QQ(x) * QQ(x) for x in v)


def public_quadratic_norm_sq(w, public_gram):
    w_q = vector(QQ, w)
    return w_q.dot_product(w_q * public_gram)


def hawk_style_norm_bound_sq(dimension, sigma_verify):
    # HAWK uses 8*n*sigma_verify^2 for vectors with dimension 2*n.
    return 4.0 * int(dimension) * (float(sigma_verify) ** 2)


def compact_signature(signature):
    return {
        "salt": signature["salt"],
        "s_vec": signature["s_vec"],
        "k": signature["k"],
        "sigma_verify": signature["sigma_verify"],
        "norm_bound_sq": signature["norm_bound_sq"],
    }


def hawk_style_e8_sign(
    msg_bits,
    sigma,
    salt=None,
    k=1,
    pubkey_bytes=None,
    secret_key=None,
    secret_basis=None,
    public_key=None,
    sigma_verify=None,
    norm_bound_sq=None,
    max_attempts=1000,
):
    if secret_key is not None:
        secret_basis = secret_key["basis"]
        public_key = secret_key.get("public_key", public_key)
        k = secret_key.get("k", k)

    if public_key is not None:
        k = public_key.get("k", k)
        public_gram = public_key["Q"]
        if pubkey_bytes is None:
            pubkey_bytes = public_key["bytes"]
        if secret_basis is None:
            raise ValueError("public_key alone is not enough to sign; pass secret_key or secret_basis")

    if secret_basis is None:
        basis = direct_sum_e8_basis(k)
        if public_key is None:
            public_gram = public_key_gram_matrix(k, basis)
        if pubkey_bytes is None:
            pubkey_bytes = public_key_gram_descriptor(k)
    else:
        basis = secret_basis
        if basis.nrows() != basis.ncols() or basis.nrows() != 8 * k:
            raise ValueError("secret_basis must be an 8k x 8k matrix")
        if public_key is None:
            public_gram = public_key_gram_matrix(k, basis)
        if pubkey_bytes is None:
            pubkey_bytes = public_key_gram_descriptor_from_matrix(k, public_gram)

    if salt is None:
        salt_b = os.urandom(16)
    else:
        salt_b = _normalize_salt(salt)
    h, _, salt_b = hash_to_binary_challenge(
        msg_bits,
        salt_b,
        pubkey_bytes=pubkey_bytes,
        k=k,
    )
    coset_rep, half_shift = challenge_to_lattice_shift(h, basis)
    bootstrap_sigma = 2 * sigma
    if sigma_verify is None:
        sigma_verify = sigma
    if norm_bound_sq is None:
        norm_bound_sq = hawk_style_norm_bound_sq(len(h), sigma_verify)

    attempts = 0
    rejected_norms = []

    while True:
        attempts += 1
        if max_attempts is not None and attempts > max_attempts:
            raise RuntimeError("rejection loop exceeded max_attempts")

        bootstrap_sample = sample_direct_sum_coset_rep_plus_2E8(bootstrap_sigma, coset_rep)
        if not in_direct_sum_rep_plus_2E8(bootstrap_sample, coset_rep):
            raise RuntimeError("sampler output does not lie in h * B_E8 + 2E8^k")

        sampled_norm_sq = squared_euclidean_norm(bootstrap_sample)
        if float(sampled_norm_sq) > float(norm_bound_sq):
            rejected_norms.append(sampled_norm_sq)
            continue

        shifted_sample = vector(QQ, [QQ(x) / 2 for x in bootstrap_sample])
        shifted_difference = shifted_sample - half_shift
        if not in_direct_sum_E8(shifted_difference):
            raise RuntimeError("halved sampler output does not lie in h * B_E8 / 2 + E8^k")

        sampled_w = lattice_point_to_basis_coefficients(bootstrap_sample, basis)
        if any((sampled_w[i] - h[i]) % 2 != 0 for i in range(len(h))):
            raise RuntimeError("sample coordinates are not congruent to h modulo 2")

        sym_break_flipped = not sym_break_vector(sampled_w)
        w = -sampled_w if sym_break_flipped else sampled_w
        sym_broken_bootstrap_sample = vector(QQ, w * basis)
        if not in_direct_sum_rep_plus_2E8(sym_broken_bootstrap_sample, coset_rep):
            raise RuntimeError("sym-broken representative does not lie in h * B_E8 + 2E8^k")

        sym_broken_norm_sq = squared_euclidean_norm(sym_broken_bootstrap_sample)
        if float(sym_broken_norm_sq) > float(norm_bound_sq):
            raise RuntimeError("sym-broken representative exceeded the norm bound")

        sym_broken_shifted_sample = vector(QQ, [QQ(x) / 2 for x in sym_broken_bootstrap_sample])
        sym_broken_shifted_difference = sym_broken_shifted_sample - half_shift
        if not in_direct_sum_E8(sym_broken_shifted_difference):
            raise RuntimeError("sym-broken half-shift representative does not lie in h * B_E8 / 2 + E8^k")
        s_vec = vector(ZZ, [(h[i] - w[i]) // 2 for i in range(len(h))])
        break

    return {
        "message_bits": _normalize_binary_message(msg_bits),
        "pubkey_bytes": _normalize_pubkey(pubkey_bytes),
        "salt": salt_b,
        "k": k,
        "b": 2,
        "public_gram": public_gram,
        "sigma": sigma,
        "sigma_verify": sigma_verify,
        "bootstrap_sigma": bootstrap_sigma,
        "norm_bound_sq": norm_bound_sq,
        "sampled_norm_sq": sampled_norm_sq,
        "sym_broken_norm_sq": sym_broken_norm_sq,
        "public_norm_sq": public_quadratic_norm_sq(w, public_gram),
        "attempts": attempts,
        "norm_rejections": attempts - 1,
        "rejected_norms": rejected_norms,
        "h": h,
        "coset_rep": coset_rep,
        "half_shift": half_shift,
        "sampled_bootstrap_sample": bootstrap_sample,
        "sampled_shifted_sample": shifted_sample,
        "sampled_shifted_sample_minus_half_shift": shifted_difference,
        "bootstrap_sample": sym_broken_bootstrap_sample,
        "shifted_sample": sym_broken_shifted_sample,
        "shifted_sample_minus_half_shift": sym_broken_shifted_difference,
        "sampled_w": sampled_w,
        "w": w,
        "s_vec": s_vec,
        "sampled_sym_break": sym_break_vector(sampled_w),
        "sym_break_flipped": sym_break_flipped,
        "sym_break": sym_break_vector(w),
        "challenge_is_binary": all(bit in (ZZ(0), ZZ(1)) for bit in h),
        "half_shift_doubles_to_rep": vector(ZZ, [ZZ(2 * x) for x in half_shift]) == coset_rep,
        "sampled_sample_lies_in_coset": in_direct_sum_rep_plus_2E8(bootstrap_sample, coset_rep),
        "sample_lies_in_coset": in_direct_sum_rep_plus_2E8(sym_broken_bootstrap_sample, coset_rep),
        "sampled_shifted_sample_lies_in_half_shift_plus_E8": in_direct_sum_E8(shifted_difference),
        "shifted_sample_lies_in_half_shift_plus_E8": in_direct_sum_E8(sym_broken_shifted_difference),
        "sampled_norm_within_bound": float(sampled_norm_sq) <= float(norm_bound_sq),
        "sym_broken_norm_within_bound": float(sym_broken_norm_sq) <= float(norm_bound_sq),
        "public_norm_within_bound": float(public_quadratic_norm_sq(w, public_gram)) <= float(norm_bound_sq),
        "w_minus_h_even": all((w[i] - h[i]) % 2 == 0 for i in range(len(h))),
        "reconstructs_w_from_signature": h - 2 * s_vec == w,
    }


def hawk_style_e8_verify(msg_bits, signature, public_key, sigma_verify=None, norm_bound_sq=None, pubkey_bytes=None):
    k = public_key["k"]
    public_gram = public_key["Q"]
    if pubkey_bytes is None:
        pubkey_bytes = public_key["bytes"]
    if sigma_verify is None:
        if "sigma_verify" not in signature:
            raise ValueError("sigma_verify must be supplied for compact signatures without parameters")
        sigma_verify = signature["sigma_verify"]

    salt_b = _normalize_salt(signature["salt"])
    h, _, _ = hash_to_binary_challenge(
        msg_bits,
        salt_b,
        pubkey_bytes=pubkey_bytes,
        k=k,
    )
    s_vec = vector(ZZ, signature["s_vec"])
    if len(s_vec) != len(h):
        return {"valid": False, "reason": "signature vector has wrong length"}

    w = h - 2 * s_vec
    if norm_bound_sq is None:
        norm_bound_sq = signature.get("norm_bound_sq", hawk_style_norm_bound_sq(len(h), sigma_verify))
    public_norm_sq = public_quadratic_norm_sq(w, public_gram)
    sym_break_ok = sym_break_vector(w)
    norm_ok = float(public_norm_sq) <= float(norm_bound_sq)
    return {
        "valid": sym_break_ok and norm_ok,
        "h": h,
        "w": w,
        "sym_break": sym_break_ok,
        "public_norm_sq": public_norm_sq,
        "norm_bound_sq": norm_bound_sq,
        "norm_within_bound": norm_ok,
    }


In [12]:
message = "10100100101101100001"
fixed_salt = bytes.fromhex("00112233445566778899aabbccddeeff")

# k = 64 gives ambient dimension 512, matches HAWK's n = 256 dimension.
k = 64
sigma = 2.2
keypair = generate_e8_hawk_style_keypair(k, steps_per_block=8)
secret_key = keypair["secret_key"]
public_key = keypair["public_key"]

signature = hawk_style_e8_sign(message, sigma=sigma, salt=fixed_salt, secret_key=secret_key)
signature_public = compact_signature(signature)
verification = hawk_style_e8_verify(message, signature_public, public_key)
print(f"parameters: k={signature['k']}, dimension={len(signature['coset_rep'])}, b={signature['b']}, sigma={signature['sigma']}")
print("public verification:", verification["valid"])
print("checks:", {
    "binary_h": signature["challenge_is_binary"],
    "half_shift": signature["half_shift_doubles_to_rep"],
    "coset_sample": signature["sample_lies_in_coset"],
    "half_shift_sample": signature["shifted_sample_lies_in_half_shift_plus_E8"],
    "w_equals_h_minus_2s": signature["reconstructs_w_from_signature"],
    "sym_break": signature["sym_break"],
    "norm_bound": signature["sym_broken_norm_within_bound"],
    "public_norm": verification["norm_within_bound"],
})
print("norm rejections:", signature["norm_rejections"])

signature_2 = hawk_style_e8_sign(message, sigma=sigma, salt=fixed_salt, secret_key=secret_key)
print("fixed transcript gives same h:", signature["h"] == signature_2["h"])
print("sampling is randomized:", signature["shifted_sample"] != signature_2["shifted_sample"])


parameters: k=64, dimension=512, b=2, sigma=2.20000000000000
public verification: True
checks: {'binary_h': True, 'half_shift': True, 'coset_sample': True, 'half_shift_sample': True, 'w_equals_h_minus_2s': True, 'sym_break': True, 'norm_bound': True, 'public_norm': True}
norm rejections: 0
fixed transcript gives same h: True
sampling is randomized: True
